In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

response = llm.invoke("hi")

print(response.content)

Hello! How can I assist you today?


In [1]:
import os, getpass
from pathlib import Path

# Hop to the project root so guardrails.hub can find ./.guardrails/hub_registry.json
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)

### Unstructured Input

In [3]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)

In [4]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

In [ ]:
from guardrails import Guard

# Guardrails calls LiteLLM under the hood. Use a LiteLLM-recognized model id
# (gpt-4.0-mini is not a real OpenAI model — that is what caused BadRequestError).
BOT_MODEL = "gpt-4o-mini"

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='4868323152',
    raw_llm_output='{\n  "customer_name": "Jane Doe",\n  "amount": 49.99,\n  "reason": "Product defect",\n  "category": "billing",\n  "urgency": "medium"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'Jane Doe',
        'amount': 49.99,
        'reason': 'Product defect',
        'category': 'billing',
        'urgency': 'medium'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
data: {'customer_name': 'Jane Doe', 'amount': 49.99, 'reason': 'Product defect', 'category': 'billing', 'urgency': 'medium'}


/Users/rishabh013/prod_new/guardrails/gaurdrails_ai/.venv/lib/python3.14/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [7]:
if data is None:
    print("No structured output yet — check the previous cell to see what the model returned.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

Customer : Jane Doe
Amount   : $49.99
Route to : billing team
Urgency  : medium
